In [24]:
import requests
import pandas as pd


from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import itertools
import matplotlib.patches as patches

import glob


# 1. Using AlphaFold API to download structures

In [ ]:
# AlphaFold Downloader for UniProt ID


def download_alphafold_pdb(uniprot_id, output_path=None):
    """
    Download the AlphaFold PDB structure for a given UniProt ID.
    
    Parameters:
    - uniprot_id (str): The UniProt ID (e.g., 'P69905')
    - output_path (str): Optional. Path to save the PDB file.
    
    Returns:
    - str: File path of the downloaded PDB file, or None if failed.
    """
    base_url = "https://alphafold.ebi.ac.uk/files"
    pdb_url = f"{base_url}/AF-{uniprot_id}-F1-model_v4.pdb"
    
    if output_path is None:
        output_path = f"{uniprot_id}.pdb"
    
    try:
        print(f"Fetching AlphaFold structure for UniProt ID: {uniprot_id}")
        response = requests.get(pdb_url)
        response.raise_for_status()
        
        with open(output_path, 'w') as file:
            file.write(response.text)
        
        print(f"✅ Download complete: {output_path}")
        return output_path
    
    except requests.HTTPError as http_err:
        print(f"❌ HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"❌ Other error occurred: {err}")
    
    return None

# Example: Downloading Hemoglobin subunit alpha (UniProt ID: P69905)
download_alphafold_pdb("P11308")


Fetching AlphaFold structure for UniProt ID: P11308
✅ Download complete: P11308.pdb


'P11308.pdb'

In [9]:
lambert_TFs = pd.read_csv("../output/lambert_TFs_10-21-24_with_DBD_coords.csv")
lambert_TFs["uniprotID"] = lambert_TFs["id"].str.split("|").str[1]
lambert_TFs

,Unnamed: 0,id,ProteinSeq,DBD_coords_merged,uniprotID
0,0,sp|A0A087WUV0|ZN892_HUMAN Zinc finger protein ...,MEPEGRGSLFEDSDLLHAGNPKENDVTAVLLTPGSQELMIRDMAEA...,"[[221, 243], [249, 271], [277, 299], [305, 327...",A0A087WUV0
1,1,sp|A0AVK6|E2F8_HUMAN Transcription factor E2F8...,MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...,"[[114, 182], [262, 347]]",A0AVK6
2,2,sp|A0PJY2|FEZF1_HUMAN Fez family zinc finger p...,MDSSCHNATTKMLATAPARGNMMSTSKPLAFSIERIMARTPEPKAL...,"[[260, 282], [288, 310], [316, 338], [344, 366...",A0PJY2
3,3,sp|A1A519|F170A_HUMAN Protein FAM170A OS=Homo ...,MKRRQKRKHLENEESQETAEKGGGMSKSQEDALQPGSTRVAKGWSQ...,"[[1, 330]]",A1A519
4,4,sp|A1YPR0|ZBT7C_HUMAN Zinc finger and BTB doma...,MANDIDELIGIPFPNHSSEVLCSLNEQRHDGLLCDVLLVVQEQEYR...,"[[364, 386], [392, 414], [420, 442], [448, 469]]",A1YPR0
...,...,...,...,...,...
1608,1608,sp|Q9Y6Q9|NCOA3_HUMAN Nuclear receptor coactiv...,MSGLGENLDPLASDSRKRKLPCDTPGQGLTCSGEKRRREQESKYIE...,"[[31, 83]]",Q9Y6Q9
1609,1609,sp|Q9Y6R6|Z780B_HUMAN Zinc finger protein 780B...,MVHGSVTFRDVAIDFSQEEWECLQPDQRTLYRDVMLENYSHLISLG...,"[[165, 187], [193, 215], [221, 243], [249, 271...",Q9Y6R6
1610,1610,sp|Q9Y6X0|SETBP_HUMAN SET-binding protein OS=H...,MESRETLSSSRQRGGESDFLPVSSAKPPAAPGCAGEPLLSTPGPGK...,"[[583, 596], [1015, 1027], [1450, 1462]]",Q9Y6X0
1611,1611,sp|Q9Y6X8|ZHX2_HUMAN Zinc fingers and homeobox...,MASKRKSTTPCMVRTSQVVEQDVPEEVDRAKEKGIGTPQPDVAKDS...,"[[78, 101], [110, 133], [271, 317], [442, 496]...",Q9Y6X8


In [10]:
activator_TFs = pd.read_csv("../soto_analysis/outputs/all_TFs_table_proteins_activators.txt", sep = "\t")
activator_TFs

,Unnamed: 0,1,2,uniprotID,ENSG,ENST,DBD_coords,AD_coords,RD_coords,Bif_coords,length
0,0,NaN,NaN,A6NJG6,NaN,ENST00000334384,79-135,142-315,152-315,NaN,1
1,1,NaN,NaN,A8MTJ6,NaN,ENST00000428390,145-234,369-420,NaN,NaN,1
2,2,NaN,NaN,A8MYZ6,NaN,ENST00000641094,89-178,382-491,NaN,NaN,1
3,3,NaN,NaN,A8MZ59,NaN,ENST00000640845,31-65,92-181,22-101,NaN,1
4,4,NaN,NaN,O00321,NaN,ENST00000403402,242-321,1-162,22-121,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...
441,441,NaN,NaN,Q9Y5R6,NaN,ENST00000382276,72-118,172-251,182-261,NaN,1
442,442,NaN,NaN,Q9Y5W3,NaN,ENST00000248071,"272-296,302-326,332-354",1-89,62-268,NaN,1
443,443,NaN,NaN,Q9Y692,NaN,ENST00000294409,89-165,422-573,"52-141,202-291",NaN,1
444,444,NaN,NaN,Q9Y6Q9,NaN,ENST00000371998,31-83,621-1424,"12-181,292-371,652-861",NaN,1


In [13]:
# This cell took about 6 minutes to run
for uniprotID in lambert_TFs["uniprotID"]:
    output_file = f"../data/lambert_TF_AF/{uniprotID}.pdb"
    download_alphafold_pdb(uniprotID, output_path=output_file)

Fetching AlphaFold structure for UniProt ID: A0A087WUV0
✅ Download complete: ../data/lambert_TF_AF/A0A087WUV0.pdb
Fetching AlphaFold structure for UniProt ID: A0AVK6
✅ Download complete: ../data/lambert_TF_AF/A0AVK6.pdb
Fetching AlphaFold structure for UniProt ID: A0PJY2
✅ Download complete: ../data/lambert_TF_AF/A0PJY2.pdb
Fetching AlphaFold structure for UniProt ID: A1A519
✅ Download complete: ../data/lambert_TF_AF/A1A519.pdb
Fetching AlphaFold structure for UniProt ID: A1YPR0
✅ Download complete: ../data/lambert_TF_AF/A1YPR0.pdb
Fetching AlphaFold structure for UniProt ID: A2RRD8
✅ Download complete: ../data/lambert_TF_AF/A2RRD8.pdb
Fetching AlphaFold structure for UniProt ID: A2RU54
✅ Download complete: ../data/lambert_TF_AF/A2RU54.pdb
Fetching AlphaFold structure for UniProt ID: A4D1E1
✅ Download complete: ../data/lambert_TF_AF/A4D1E1.pdb
Fetching AlphaFold structure for UniProt ID: A6NCS4
✅ Download complete: ../data/lambert_TF_AF/A6NCS4.pdb
Fetching AlphaFold structure for UniPr

# 2. Running DSSP 

In [ ]:
# G 3-turn helix (3_10 helix)
# H 4-turn helix (alpha-helix)
# I 5-turn helix (pi helix)

# E extended strand in parallel and/or anti-parallel β-sheet conformation. Min length 2 residues
# B residue in isolated β-bridge (single pair β-sheet hydrogen bond formation)

# P Polyproline

# T H-bonded turn
# S bend

# - coil -- ignore

In [34]:
from concurrent.futures import ProcessPoolExecutor

In [55]:
def parse_pdb(path):
    try:
        uniprotID = path.split("/")[-1].split(".")[0]
        p = PDBParser()
        structure = p.get_structure(uniprotID, path)
        model = structure[0]
        dssp = DSSP(model, path)
        pos = []
        code = []
        
        keys = list(dssp.keys())
        for i in range(len(keys)):
            pos.append(i)
            a_key = keys[i]
            code.append(dssp[a_key][2])

        codes = pd.DataFrame({"pos" : pos, "code" : code})
        codes["uniprotID"] = uniprotID
        return codes
    except Exception as e: 
        print(f"Error parsing {uniprotID}: {e}")

In [56]:
parse_pdb("../data/lambert_TF_AF/P11308.pdb")

,pos,code,uniprotID
0,0,-,P11308
1,1,-,P11308
2,2,-,P11308
3,3,-,P11308
4,4,-,P11308
...,...,...,...
474,474,-,P11308
475,475,-,P11308
476,476,-,P11308
477,477,-,P11308


In [57]:
from concurrent.futures import ThreadPoolExecutor

paths = glob.glob("../data/lambert_TF_AF/*")

with ThreadPoolExecutor() as executor:
    dssp_dfs = list(executor.map(parse_pdb, paths))

/opt/anaconda3/lib/python3.12/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Trying to parse 'HUMA'
Error parsing PDB at line 38
Not a valid integer in PDB record

  warnings.warn(err)


Error parsing A0A087WUV0: DSSP failed to produce an output


In [58]:
dssp_df = pd.concat(dssp_dfs)
dssp_df

,pos,code,uniprotID
0,0,-,Q9NZC4
1,1,-,Q9NZC4
2,2,-,Q9NZC4
3,3,-,Q9NZC4
4,4,-,Q9NZC4
...,...,...,...
454,454,S,Q8TBJ5
455,455,S,Q8TBJ5
456,456,S,Q8TBJ5
457,457,-,Q8TBJ5


In [59]:
dssp_df.to_csv("../output/lambert_TF_AF_dssp.csv")

In [60]:
len(set(dssp_df["uniprotID"]))

1600